# Scout in 10 minutes

A guided tour of what's in the lake, running entirely on Trino so every cell returns in seconds.

Scout ingests HL7 radiology messages into a Delta Lake `reports` table on MinIO. The columns mix structured fields (modality, dates, demographics, diagnosis codes) with the full free-text report. Helpers live in [`scout_demo.py`](scout_demo.py); this notebook is the narrative around them.

> Run cells top-to-bottom. Each section stands alone.

## 1. Connect

One import. `q(sql)` runs a Trino query and returns a pandas DataFrame.

In [ ]:
from scout_demo import (
    q,
    kpi_cards,
    plot_modality_volume,
    plot_volume_trend,
    plot_demographics,
    search_text,
    plot_top_diagnoses,
    plot_tat,
)

## 2. Five rows from the lake

`q(sql)` returns a pandas DataFrame, which Jupyter renders inline. Every notebook trick — filtering, joining, exporting to CSV — starts here.

In [ ]:
q("""
    SELECT modality, sex, patient_age, service_name,
           CAST(requested_dt AS DATE) AS requested
    FROM reports_latest
    LIMIT 5
""")

## 3. Scale at a glance

Topline counts over the whole table. The numbers don't drop reports just because a single date column happens to be null.

In [ ]:
kpi_cards()

## 4. What kind of imaging?

Modality is derived during the HL7 transform from the OBR/MSH segments.

In [ ]:
plot_modality_volume()

## 5. Volume over time

Last five years, monthly, stacked by modality. Look for seasonality, the COVID dip, or a facility that came online partway through.

In [ ]:
plot_volume_trend(years=5)

## 6. Who shows up where?

Each cell is the share of that modality falling in the age band — columns sum to 100%.

In [ ]:
plot_demographics()

## 7. Free-text search at scale

`REGEXP_LIKE` runs across millions of report bodies in seconds. Change `pattern` to anything you want — *pneumonia*, *aneurysm*, *follow-up recommended*. The helper returns a DataFrame so pandas can take it from there.

In [ ]:
pattern = "pulmonary embolism"  # change me
matches = search_text(pattern)
matches

## 8. Structured diagnoses, not just text

The `diagnoses` column is an array of structs from the HL7 DG1 segments. `UNNEST` turns it into one row per code so you can rank them — or filter for specific ones with `any_match(diagnoses, e -> e.diagnosis_code IN (…))`.

In [ ]:
plot_top_diagnoses(top_n=15)

## 9. Operational signal: turnaround time

Order placed → final report signed, in hours. Median and 90th percentile per modality — a quick read on where the slowest tails live.

In [ ]:
plot_tat()

## 10. Where to go from here

Same Delta Lake + Trino backbone powers Scout's other surfaces:

- **Cohort Builder** (`analytics/notebooks/cohort/Cohort.ipynb`) — point-and-click cohort assembly with diagnosis codes, free text, and demographic filters; exports CSV/Parquet.
- **Quality Metrics** (`analytics/notebooks/quality-metrics/QualityMetrics.ipynb`) — TAT, addendum rates, and report-status drilldowns as a Voilà playbook.
- **Follow-Up Detection** (`analytics/notebooks/followup-detection/FollowUpDetection.ipynb`) — LLM classification of which reports recommend patient-specific follow-up, plus a reviewer dashboard.
- **Superset** — every chart here can be a no-code dashboard against the same `delta.default.reports` table.

Same data, same queries, different surfaces — pick the one that matches the user.